# Unused functions for MAE3405 Asst1 

In [1]:
def fuelAirDieselCycle(T1,rho1,RPM,bore=0.1,stroke=0.1,ncyl=4,compressionRatio=15,\
                     LHV=42.5e3,AFstoich=14.5,equivalenceRatio=1.,combustionEfficiency=1.,\
                    mechanicalEfficiency=.95,fourStroke=True,volumetricEfficiency=1.,\
                      cutoffRatio=2.6):
    
    # Calculate the performance of a Fuel-air Diesel Cycle.
    # Assumed fuel and air already mixed at intake (non DI)
    # No chemistry calcualtion - provide LHV, AFs & combustion efficiency as inputs.
    # No super/turbocharging calculation - provide T1 and p1 as inputs.
    
    # Temps in Kelvin.
    # Bore, stroke in m
    # LHV in kJ/kg_f

    # Apply some sanity checks to inputs
    mechanicalEfficiency = boundEfficiency(mechanicalEfficiency)
    volumetricEfficiency = boundEfficiency(volumetricEfficiency)
    combustionEfficiency = boundEfficiency(combustionEfficiency)
    assert all((T1>0, rho1>0, bore>0, stroke>0, ncyl>0, compressionRatio>0))
    assert all((LHV>0, AFstoich>0, equivalenceRatio>0))
    
    
    # Thermo constants for Fuel-Air Cycle from notes
    gamma_r = 1.38
    gamma_p = 1.25
    R_r = 0.266 #kJ/kg.K
    R_p = 0.275 #kJ/kg.K
    cp_r = (gamma_r*R_r)/(gamma_r-1) #kJ/kg.K
    cp_p = (gamma_p*R_p)/(gamma_p-1) #kJ/kg.K
    cv_r = (R_r)/(gamma_r-1) #kJ/kg.K
    cv_p = (R_p)/(gamma_p-1) #kJ/kg.K
    
    # Fuel-air cycle
    AF = AFstoich/equivalenceRatio
    Qh = combustionEfficiency * LHV / (1+AF)

    p1 = rho1*R_r*T1*1e3 # Pa
    v1 = 1/rho1 # m3/kg
    
    p2 = p1*(compressionRatio)**(gamma_r)   # isentropic compression
    T2 = T1*(compressionRatio)**(gamma_r-1) # isentropic compression
    v2 = v1/compressionRatio                # isentropic compression
    
    v3 = v2*cutoffRatio                  # isobaric expansion
    w23 = p2*(v3-v2)*1e-3  #kJ/kg        # isobaric expansion
    T3 = (cp_r*(T2-298) + Qh - w23)/cp_p + 298 # isobaric heating
    p3 = R_p*T3/v3

    v4 = v1 # mass and volume back to same as state 1 if non-DI
    p4 = p3*(v3/v4)**gamma_p # note that expansion ratio ≠ compression ratio in Diesel
    T4 = T3*(v3/v4)**(gamma_p-1) # isentropic expansion
    
    Ql = cv_p*(T4-298)-cv_r*(T1-298)
    eta_th = (Qh-Ql)/Qh
    eta_bth = mechanicalEfficiency * eta_th

    # Engine volume and power
    sweptVol = ncyl*stroke*(bore**2)*np.pi*0.25 # m^3
    sweptRate = sweptVol * (RPM/60.) # m^3/s
    if fourStroke: sweptRate /= 2.
    inducedAirRate = rho1*sweptRate*volumetricEfficiency # kg/s
    brakePower = (LHV*1e3/AF)*inducedAirRate*eta_bth # Watts

    # Fuel consumption
    fuelRate = inducedAirRate/AF
    
    # Performance
    BMEP = brakePower/sweptRate/1e6 # MPa
    BSFC = fuelRate / brakePower * 1e3 * 3600 #kg/kW.hr

    return eta_bth, brakePower, BMEP, BSFC

In [ ]:
from scipy.optimize import root

def propulsiveEfficiency_Froude(airspeed=100., propDiameter=2.0, shaftPower=100e3, rho=1.2):
    # Solve the Froude Theory equations for efficiency and thrust given power input
    
    # Return the efficiency and thrust produced by an ideal propeller
    A = 0.25*np.pi*propDiameter**2
    u = airspeed
    
    def fFroude(vars):
        w, thrust = vars # w is induced velocity m/s
        return [ thrust*(u+w) - shaftPower ,\
                 -2*w - u + np.sqrt(u**2 + 2*thrust/(rho*A)) ]

    # Use static thrust for initial guess
    guess_thrust = np.cbrt( shaftPower**2 * (2*rho*A) )
    guess_w = np.sqrt(guess_thrust/(2*rho*A))
    
    solveFroude = root(fFroude, [guess_w,guess_thrust])
    w, thrust = solveFroude.x
    propulsiveEfficiency=1/(1+w/u)

    return propulsiveEfficiency, thrust